# FixtureIQ — Updated Data Exploration: All Sources (2022-23 to 2024-25)

## Objectives
1. Inventory every data source available in the project using the new consolidated layouts
2. Check missing values, data quality, schema compatibility
3. Understand name matching across new data frames (SofaScore vs FBref Master vs Injury)
4. Assess cross-source merge feasibility
5. Build a unified view of what data is usable

## Seasons Covered: 2022-23, 2023-24, 2024-25

## Data Sources
- **SofaScore Dynamic** (`Data/<season>/sofascore_dynamic/`) — per-match player stats + workload features
- **FBref Team Masters** (`Data/<season>/fbref/<team_folder>/master_*.csv`) — consolidated multi-season match reports
- **SofaScore Premier League aggregates** (`Data/<season>/sofascore/premier_league/`) — season-level player stats
- **Injuries** (`Data/<season>/injuries/`) — injury records for all PL teams
- **Club Elo** (`Data/clubelo_understat/fixtureiq_elo_master.csv`) — team strength ratings
- **Understat** (`Data/clubelo_understat/fixtureiq_understat_master.csv`) — xG match data

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os, json, re, glob
from pathlib import Path
from collections import defaultdict
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 5)

BASE = Path('..')
print(f'Project root: {BASE.resolve()}')
print(f'Data folder exists: {(BASE / "Data").exists()}')

Project root: C:\Users\ItxaroAizpuruaArcona\Desktop\S_A\Fixture-IQ-playground
Data folder exists: True


---
## 1. SofaScore Dynamic Data (Primary Workload Dataset)

Per-match player records with pre-engineered workload features: `rest_days`, `acwr_ratio`, `high_congestion_flag`, `min_last_7d`, etc.

In [4]:
def explore_sofascore_dynamic(season_code):
    path = BASE / 'Data' / season_code / 'sofascore_dynamic' / 'fixtureiq_dynamic_analytics_clean.csv'
    master_path = BASE / 'Data' / season_code / 'sofascore_dynamic' / 'fixtureiq_dynamic_master.csv'
    
    if not path.exists():
        print(f'[WARN] {season_code}: clean file not found')
        return None, None
    
    df = pd.read_csv(path)
    df_master = pd.read_csv(master_path) if master_path.exists() else None
    
    print(f'=== {season_code} SofaScore Dynamic ===')
    print(f'Clean: {df.shape[0]:,} rows x {df.shape[1]} cols')
    if df_master is not None:
        print(f'Master: {df_master.shape[0]:,} rows x {df_master.shape[1]} cols')
    
    date_col = 'match_date_str' if 'match_date_str' in df.columns else 'date'
    print(f'Date range: {df[date_col].min()} to {df[date_col].max()}')
    
    nulls = df.isnull().sum()
    nulls = nulls[nulls > 0]
    if len(nulls) > 0:
        print(f'Missing values:')
        for c, v in nulls.items():
            print(f'  {c}: {v:>6} ({v/df.shape[0]*100:5.1f}%)')
    else:
        print('Missing values: (none)')
    
    print(f'Unique teams: {df["teamName"].nunique()}')
    print(f'Unique players: {df["name"].nunique()}')
    
    if 'minutesPlayed' in df.columns:
        no_min = (df['minutesPlayed'] == 0).sum()
        low_min = ((df['minutesPlayed'] > 0) & (df['minutesPlayed'] <= 45)).sum()
        print(f'Minutes distribution:')
        print(f'  0 min (unused sub): {no_min:>6} ({no_min/df.shape[0]*100:5.1f}%)')
        print(f'  1-45 min (sub):     {low_min:>6} ({low_min/df.shape[0]*100:5.1f}%)')
    
    return df, df_master

sofascore_dynamic = {}
for s in ['2022-2023', '2023-2024', '2024-2025']:
    df, dfm = explore_sofascore_dynamic(s)
    if df is not None:
        sofascore_dynamic[s] = df

=== 2022-2023 SofaScore Dynamic ===
Clean: 6,500 rows x 19 cols
Master: 7,304 rows x 108 cols
Date range: 2022-08-06 to 2023-06-10
Missing values:
  rating:   1616 ( 24.9%)
  elo:    707 ( 10.9%)
  team_xg:   6500 (100.0%)
  team_xga:   6500 (100.0%)
  xg_difference:   6500 (100.0%)
Unique teams: 36
Unique players: 900
Minutes distribution:
  0 min (unused sub):   1488 ( 22.9%)
  1-45 min (sub):       1515 ( 23.3%)
=== 2023-2024 SofaScore Dynamic ===
Clean: 13,142 rows x 17 cols
Master: 14,758 rows x 106 cols
Date range: 2023-08-11 to 2024-04-25
Missing values:
  rating:   3130 ( 23.8%)
Unique teams: 34
Unique players: 938
Minutes distribution:
  0 min (unused sub):   2910 ( 22.1%)
  1-45 min (sub):       3123 ( 23.8%)
=== 2024-2025 SofaScore Dynamic ===
Clean: 6,726 rows x 19 cols
Master: 7,587 rows x 117 cols
Date range: 2024-08-17 to 2025-05-25
Missing values:
  rating:   1592 ( 23.7%)
  elo:    678 ( 10.1%)
  team_xg:   6726 (100.0%)
  team_xga:   6726 (100.0%)
  xg_difference:   6

---
## 2. FBref Per-Match Data (Consolidated Team Masters Layout)

Evaluating the newly organized layout where match data is bundled by team folder into `master_player_stats.csv`, `master_goalkeeper_stats.csv`, and `master_lineups.csv` files.

In [7]:
def explore_fbref_season_consolidated(season_code):
    """Explore the new fast-loading consolidated team master files."""
    base = BASE / 'Data' / season_code / 'fbref'
    if not base.exists():
        print(f'{season_code}: no FBref directory')
        return {}
    
    teams = {}
    for team_dir in sorted(base.iterdir()):
        if not team_dir.is_dir():
            continue
        
        master_ps = team_dir / 'match_reports' / 'master_player_stats.csv'
        master_gk = team_dir / 'match_reports' / 'master_goalkeeper_stats.csv'
        master_lu = team_dir / 'match_reports' / 'master_lineups.csv'
        
        teams[team_dir.name] = {
            'has_player_stats': master_ps.exists(),
            'has_gk_stats': master_gk.exists(),
            'has_lineups': master_lu.exists(),
            'player_stats_rows': pd.read_csv(master_ps).shape[0] if master_ps.exists() else 0
        }
    return teams

for s in ['2022-2023', '2023-2024', '2024-2025']:
    teams = explore_fbref_season_consolidated(s)
    if teams:
        print(f'=== {s} FBref (Consolidated Structure) ===')
        for t, info in teams.items():
            print(f'  {t}: Stats={info["player_stats_rows"]} rows | GK={info["has_gk_stats"]} | Lineups={info["has_lineups"]}')

    print("\n")

=== 2022-2023 FBref (Consolidated Structure) ===
  chelsea_2022_2023: Stats=778 rows | GK=True | Lineups=True
  liverpool_2022_2023: Stats=805 rows | GK=True | Lineups=True
  manchester_city_2022_2023: Stats=872 rows | GK=True | Lineups=True
  tottenham_hotspur_2022_2023: Stats=732 rows | GK=True | Lineups=True


=== 2023-2024 FBref (Consolidated Structure) ===
  arsenal_2023_2024: Stats=769 rows | GK=True | Lineups=True
  manchester_city_2023_2024: Stats=805 rows | GK=True | Lineups=True
  manchester_united_2023_2024: Stats=0 rows | GK=False | Lineups=False
  newcastle_united_2023_2024: Stats=759 rows | GK=True | Lineups=True


=== 2024-2025 FBref (Consolidated Structure) ===
  arsenal_2024_2025: Stats=866 rows | GK=True | Lineups=True
  aston_villa_2024_2025: Stats=880 rows | GK=True | Lineups=True
  liverpool_2024_2025: Stats=861 rows | GK=True | Lineups=True
  manchester_city_2024_2025: Stats=829 rows | GK=True | Lineups=True




### 2.1 FBref Data Collection Pipeline Simulation

In [13]:
def load_fbref_compiled_master():
    """Simulate loading all seasonal team master files into a collective frame."""
    all_dfs = []
    for season_str in ['2022-2023', '2023-2024', '2024-2025']:
        season_path = BASE / 'Data' / season_str / 'fbref'
        if season_path.exists():
            for team_dir in sorted(season_path.iterdir()):
                if not team_dir.is_dir():
                    continue
                ps_file = team_dir / 'match_reports' / 'master_player_stats.csv'
                if ps_file.exists():
                    df = pd.read_csv(ps_file)
                    all_dfs.append(df)
                    
    return pd.concat(all_dfs, ignore_index=True) if all_dfs else pd.DataFrame()

fbref_inventory = load_fbref_compiled_master()
print(f'Total compiled player stats entries: {fbref_inventory.shape[0]:,}')
if not fbref_inventory.empty:
    print(f'Breakdown by Season Tracking Column:')
    print(fbref_inventory.groupby('season')['match_date'].count().to_string())
    print(f'Breakdown by Tracked Team Column:')
    print(fbref_inventory.groupby('tracked_team')['match_date'].count().to_string())

Total compiled player stats entries: 8,956
Breakdown by Season Tracking Column:
season
2022-2023    3187
2023-2024    2333
2024-2025    3436
Breakdown by Tracked Team Column:
tracked_team
Arsenal 2023 2024              769
Arsenal 2024 2025              866
Aston Villa 2024 2025          880
Chelsea 2022 2023              778
Liverpool 2022 2023            805
Liverpool 2024 2025            861
Manchester City 2022 2023      872
Manchester City 2023 2024      805
Manchester City 2024 2025      829
Newcastle United 2023 2024     759
Tottenham Hotspur 2022 2023    732


---
## 3. SofaScore Season Aggregate Data (Premier League Focus Only)

Isolating data specifically belonging to the `premier_league` tracking subfolder.

In [14]:
def explore_sofascore_pl_aggregates():
    """Strictly check Premier League season-level files."""
    pattern = str(BASE / 'Data' / '*' / 'sofascore' / 'premier_league' / '*_all_players.csv')
    files = sorted(glob.glob(pattern))
    
    print(f'Found {len(files)} Premier League aggregate player files:')
    info = []
    for f in files:
        df = pd.read_csv(f)
        parts = Path(f).relative_to(BASE).parts
        season = parts[1]
        comp = parts[3]
        info.append({'file': f, 'season': season, 'comp': comp, 'players': df.shape[0], 'cols': df.shape[1]})
        print(f'  {season}/{comp}: {df.shape[0]:>4} players, {df.shape[1]:>3} cols')
    return pd.DataFrame(info)

ss_pl_agg = explore_sofascore_pl_aggregates()

Found 5 Premier League aggregate player files:
  2020-2021/premier_league:  524 players, 115 cols
  2021-2022/premier_league:  538 players, 115 cols
  2022-2023/premier_league:  554 players, 115 cols
  2023-2024/premier_league:  570 players, 115 cols
  2024-2025/premier_league:  562 players, 115 cols


---
## 4. Injury Data Verification

Validating targets and coverage flags for historical data arrays.

In [15]:
def explore_injuries():
    all_inj = []
    for s in ['2022-2023', '2023-2024', '2024-2025']:
        inj_dir = BASE / 'Data' / s / 'injuries'
        if not inj_dir.exists():
            continue
        for f in sorted(inj_dir.iterdir()):
            if f.name.endswith('_injuries_days_out.csv'):
                df = pd.read_csv(f)
                df['season'] = s
                all_inj.append(df)
    
    if not all_inj: return pd.DataFrame()
    inj = pd.concat(all_inj, ignore_index=True)
    inj['fixture_date'] = pd.to_datetime(inj['fixture_date'], errors='coerce')
    inj['days_out'] = pd.to_numeric(inj['days_out'], errors='coerce')
    return inj

injuries = explore_injuries()
if not injuries.empty:
    print(f'Total injury records found: {injuries.shape[0]:,}')
    print(injuries.groupby('season').agg({'player_name': 'nunique', 'team_name': 'nunique'}))

Total injury records found: 9,526
           player_name  team_name
season                           
2022-2023          344         20
2023-2024          404         20
2024-2025          394         20


---
## 5. Name Matching Across Sources (New Architecture Test)

Evaluating full text and alignment matches across SofaScore Dynamic, the new consolidated FBref matrix, and the historical Injury logs.

In [16]:
dyn_players = set()
for s, df in sofascore_dynamic.items():
    if 'name' in df.columns:
        dyn_players.update(df['name'].unique())

fbref_players = set()
if not fbref_inventory.empty and 'Player' in fbref_inventory.columns:
    fbref_players.update(fbref_inventory['Player'].unique())

injury_players = set(injuries['player_name'].unique()) if not injuries.empty else set()

print(f'SofaScore Dynamic unique names: {len(dyn_players):,}')
print(f'FBref Consolidated unique names: {len(fbref_players):,}')
print(f'Injury dataset unique names:     {len(injury_players):,}')

SofaScore Dynamic unique names: 1,816
FBref Consolidated unique names: 257
Injury dataset unique names:     709


---
## 6. Global Structural Integrity & Availability Matrix

In [17]:
print('=== New Structure Integrity Summary ===')
print(f'SofaScore Dynamic Observations: {sum(df.shape[0] for df in sofascore_dynamic.values()):,}')
print(f'FBref Match Records Rows:      {fbref_inventory.shape[0]:,}')
print(f'SofaScore PL Aggregates Loaded: {ss_pl_agg.shape[0]} seasonal master sheets')
print(f'Total Injury Entries Tracked:  {injuries.shape[0]:,}')

=== New Structure Integrity Summary ===
SofaScore Dynamic Observations: 26,368
FBref Match Records Rows:      8,956
SofaScore PL Aggregates Loaded: 5 seasonal master sheets
Total Injury Entries Tracked:  9,526
